In [1]:

# Import the requests library to send HTTP requests to the weather API.
import requests

import pandas as pd
from datetime import date, timedelta


# STEP 1: DEFINE THE LOCATION

# Latitude of Mumbai, India.
# Latitude identifies the north-south position of a location.
LATITUDE = 19.0760

# Longitude of Mumbai, India.
# Longitude identifies the east-west position of a location.
LONGITUDE = 72.8777



In [2]:

# STEP 2: CALCULATE THE DATE RANGE

# Get today's date according to the computer running this script.
today = date.today()

# Subtract one day from today to get yesterday.
# We exclude today because the current day's weather data
# may be incomplete.
end_date = today - timedelta(days=1)

# Subtract 29 more days from the end date.
# Including both start_date and end_date gives 30 days.
start_date = end_date - timedelta(days=29)


In [3]:

# Convert the dates into strings in YYYY-MM-DD format.
# The API expects dates such as "2026-09-16".
start_date_string = start_date.isoformat()
end_date_string = end_date.isoformat()

# Display the selected date range.
print("Start date:", start_date_string)
print("End date:", end_date_string)


Start date: 2026-09-16
End date: 2026-10-15


In [4]:
# STEP 3: DEFINE THE API ENDPOINT

# Open-Meteo's historical archive endpoint provides
# historical weather observations for a specified location
# and date range.
url = "https://archive-api.open-meteo.com/v1/archive"


# STEP 4: DEFINE THE API PARAMETERS

# A dictionary containing the inputs sent to the API.
params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,

    # Beginning of the historical date range.
    "start_date": start_date_string,

    # End of the historical date range.
    "end_date": end_date_string,

    # Daily weather variables we want the API to return.
    # Each variable is requested by its documented API name.
    "daily": [
        "temperature_2m_mean",       # Daily mean temperature in °C
        "temperature_2m_max",        # Daily maximum temperature in °C
        "temperature_2m_min",        # Daily minimum temperature in °C
        "relative_humidity_2m_mean", # Daily mean relative humidity in %
        "precipitation_sum",         # Total daily precipitation in mm
        "wind_speed_10m_max"         # Maximum daily wind speed in km/h
    ],

    # Request dates and times in Indian Standard Time.
    # This avoids interpreting daily timestamps in UTC.
    "timezone": "Asia/Kolkata"
}


In [5]:

# STEP 5: SEND THE API REQUEST

# Send an HTTP GET request to the API.
# The params dictionary is automatically converted into
# URL query parameters.
# timeout=30 means the request will wait up to 30 seconds.
response = requests.get(
    url,
    params=params,
    timeout=30
)

# Check whether the server returned an HTTP error.
# For example, this raises an exception if the server
# returns a 400 or 500 status code.
response.raise_for_status()

# Convert the response body from JSON text into a
# Python dictionary.
data = response.json()


In [6]:

# STEP 6: EXTRACT THE DAILY WEATHER DATA

# The API returns daily data inside the "daily" section.
# For example, this section contains a list of dates and
# a list of values for each requested weather variable.
daily_data = data["daily"]

# Convert the dictionary of daily lists into a pandas
# DataFrame, where each row represents one day.
df = pd.DataFrame(daily_data)



# STEP 7: RENAME THE COLUMNS

# Replace the API's technical variable names with
# readable column names for analysis and teaching.
df.rename(columns={

    "time": "Date",
    "temperature_2m_mean": "Temperature_Mean_C",
    "temperature_2m_max": "Temperature_Max_C",
    "temperature_2m_min": "Temperature_Min_C",
    "relative_humidity_2m_mean": "Humidity_Mean_Percent",
    "precipitation_sum": "Precipitation_mm",
    "wind_speed_10m_max": "Wind_Speed_Max_kmh"

}, inplace=True)  # Modify the existing DataFrame.


# STEP 8: SAVE THE DATA TO A CSV FILE

# Export the DataFrame to a CSV file.
# index=False prevents pandas from adding an extra
# numerical index column to the output file.
df.to_csv(
    "mumbai_weather_last_30_days.csv",
    index=False
)


# Display the first five rows of the collected data.
print("\nFirst five records:")
df.head()
# Display the column names and data types.
print("\nData information:")
df.info()



First five records:

Data information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Date                   30 non-null     object 
 1   Temperature_Mean_C     30 non-null     float64
 2   Temperature_Max_C      30 non-null     float64
 3   Temperature_Min_C      30 non-null     float64
 4   Humidity_Mean_Percent  30 non-null     int64  
 5   Precipitation_mm       30 non-null     float64
 6   Wind_Speed_Max_kmh     30 non-null     float64
dtypes: float64(5), int64(1), object(1)
memory usage: 1.8+ KB


In [7]:

import requests
import pandas as pd
from datetime import date, timedelta

# Mumbai coordinates
LATITUDE = 19.0760
LONGITUDE = 72.8777

# Previous 30 complete days
end_date = date.today() - timedelta(days=1)
start_date = end_date - timedelta(days=29)

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": start_date.isoformat(),
    "end_date": end_date.isoformat(),
    "daily": [
        "temperature_2m_mean",
        "temperature_2m_max",
        "temperature_2m_min",
        "relative_humidity_2m_mean",
        "precipitation_sum",
        "wind_speed_10m_max"
    ],
    "timezone": "Asia/Kolkata"
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

data = response.json()

# Convert API response into a DataFrame
df = pd.DataFrame(data["daily"])

df.rename(columns={
    "time": "Date",
    "temperature_2m_mean": "Temperature_Mean_C",
    "temperature_2m_max": "Temperature_Max_C",
    "temperature_2m_min": "Temperature_Min_C",
    "relative_humidity_2m_mean": "Humidity_Mean_Percent",
    "precipitation_sum": "Precipitation_mm",
    "wind_speed_10m_max": "Wind_Speed_Max_kmh"
}, inplace=True)

df.to_csv("mumbai_weather_last_30_days.csv", index=False)

print(df.head())
print("Rows collected:", len(df))
print("Saved to mumbai_weather_last_30_days.csv")


         Date  Temperature_Mean_C  Temperature_Max_C  Temperature_Min_C  \
0  2026-09-16                27.4               29.8               25.3   
1  2026-09-17                28.0               29.9               26.0   
2  2026-09-18                27.3               29.2               25.8   
3  2026-09-19                27.4               30.3               24.5   
4  2026-09-20                27.8               30.4               25.2   

   Humidity_Mean_Percent  Precipitation_mm  Wind_Speed_Max_kmh  
0                     83               2.8                13.5  
1                     80               1.8                14.3  
2                     84               7.1                12.8  
3                     83              10.6                13.8  
4                     82               1.1                13.1  
Rows collected: 30
Saved to mumbai_weather_last_30_days.csv
